# 3a – Erweiterter RAG-Graph: Reranking & Relevanz-Gate

In Notebook 2 war der Graph linear: `retrieve → generate → END`. Wer das Mermaid-Diagramm
aufmerksam angesehen hat, dem ist aufgefallen: **das hätte man auch als einfache Chain
lösen können.**

Hier wird LangGraph zum ersten Mal wirklich gebraucht:

- **Reranking** – ein Cross-Encoder bewertet die gefundenen Chunks neu
- **Relevanz-Gate** – eine bedingte Kante entscheidet: *sind die Chunks gut genug?*
- **Query-Reformulierung** – wenn nicht, wird die Suchanfrage umformuliert und erneut gesucht

Der Graph hat damit **bedingte Kanten** und einen **Zyklus** – beides kann eine lineare
Chain nicht.

```
retrieve → rerank → check_relevance ─(relevant)─→ generate → END
                          │
                    (nicht relevant)
                          │
                          ▼
                     reformulate ──→ retrieve   (Zyklus, max. MAX_RETRIES×)
```

> **Voraussetzung:** Notebook 1 (Indexing) muss vorher ausgeführt worden sein.

## Benötigte Pakete

```bash
pip install -U langchain-core langchain-community langchain-openai langchain-chroma \
               langgraph gradio sentence-transformers torch
# nur bei EMBEDDING_BACKEND = "huggingface":
pip install -U "langchain-huggingface[full]"
```

## Konfiguration

Wortgleich mit Notebook 1 und 2. Abschnitte, die dieses Notebook nicht braucht, stören hier nicht.

In [ ]:
# ============================================================
#  KONFIGURATION – in allen Notebooks der Reihe identisch
# ============================================================

# --- 1) Embedding-Backend -----------------------------------
#     Indexierung und Retrieval MÜSSEN dasselbe Backend nutzen.
EMBEDDING_BACKEND = "lmstudio"        # "lmstudio" | "huggingface"

# --- 2) LLM-Backend (ab Notebook 2) --------------------------
LLM_BACKEND = "lmstudio"              # "lmstudio" | "deepinfra" | "openrouter"

# --- 3) API-Keys der Cloud-Anbieter --------------------------
#     Nur ausfüllen, wenn das jeweilige Backend genutzt wird.
#     Bitte den Key für dich behalten: nicht weitergeben, nicht committen,
#     nicht in geteilten Notebooks stehen lassen.
DEEPINFRA_API_KEY  = ""
OPENROUTER_API_KEY = ""

# --- 4) Pfade ------------------------------------------------
DOC_SOURCE_DIR = "./documents"        # Quell-Dokumente (PDF + DOCX)
DB_DIR         = "./chroma_db"        # Vektordatenbank
BM25_DIR       = "./bm25_index"       # Lexikalischer Index
MODEL_PATH     = "./models"           # Modell-Cache (nur HuggingFace)
MANIFEST_PATH  = "./index_manifest.json"

COLLECTION_NAME = "langchain"         # muss in allen Notebooks gleich sein

# --- 5) Chunking (Notebook 1) --------------------------------
#     Ändert man das hier, muss der komplette Index neu gebaut werden.
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128                   # ~25 % Überlappung – gut für lange deutsche Sätze

# --- 6) Welche Indizes bauen? (Notebook 1) -------------------
BUILD_VECTOR_INDEX = True             # semantisch, braucht das Embedding-Modell
BUILD_BM25_INDEX   = True             # lexikalisch, braucht kein Modell

# --- 7) Einfaches Retrieval (Notebook 2) ---------------------
TOP_K = 10                            # Anzahl der Chunks pro Frage

# --- 8) Reranking & Relevanz-Gate (ab Notebook 3) ------------
RERANKER_MODEL      = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
RETRIEVE_K          = 15              # Kandidaten aus der Suche (breit suchen)
RERANK_TOP_N        = 5               # davon die N besten behalten
RELEVANCE_THRESHOLD = 0.3             # Mindest-Score des besten Chunks (0…1)
MAX_RETRIES         = 2               # max. Query-Reformulierungen

# --- 9) Hybrid Search (Notebook 3c) --------------------------
RETRIEVER_K      = 10                 # Kandidaten je Retriever vor der Fusion
ENSEMBLE_WEIGHTS = [0.5, 0.5]         # [Vektor, BM25] – Summe 1.0

# ============================================================
#  Backend-Details – normalerweise unverändert lassen
# ============================================================

LM_STUDIO_URL = "http://localhost:1234/v1"

EMBEDDING_MODELS = {
    "huggingface": "intfloat/multilingual-e5-large-instruct",
    "lmstudio":    "text-embedding-multilingual-e5-large-instruct",
}

LLM_CONFIG = {
    "lmstudio": {
        "base_url": LM_STUDIO_URL,
        "model":    "qwen/qwen3.5-9b",
        "api_key":  "lm-studio",                  # LM Studio prüft den Key nicht
    },
    "deepinfra": {
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "api_key":  DEEPINFRA_API_KEY,
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "model":    "meta-llama/llama-3.3-70b-instruct",
        "api_key":  OPENROUTER_API_KEY,
    },
}

print(f"Embeddings: {EMBEDDING_BACKEND}  |  LLM: {LLM_BACKEND}")

## Imports

In [ ]:
import os
import json
import base64
from typing import List, TypedDict

import torch
import gradio as gr
from IPython import display

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END

# Cross-Encoder für das Reranking
from sentence_transformers import CrossEncoder

## Womit wurde der Index gebaut?

Notebook 1 hinterlegt ein Manifest. Diese Zelle **gibt es nur aus** – sie prüft nichts.

> ⚠️ Vergleiche `embedding_backend` und `embedding_modell` mit deiner Konfiguration oben.

In [ ]:
if os.path.exists(MANIFEST_PATH):
    with open(MANIFEST_PATH, encoding="utf-8") as f:
        manifest = json.load(f)

    print(f"📝 Index-Manifest ({MANIFEST_PATH}):")
    for key, value in manifest.items():
        print(f"   {key:<18} {value}")
else:
    print(f"ℹ️  Kein Manifest unter '{MANIFEST_PATH}' gefunden.")
    print("   Der Index stammt vermutlich aus einem älteren Lauf von Notebook 1.")

## Embedding-Modell & Vektordatenbank

Identisch zu Notebook 1 und 2: E5 erwartet die Präfixe `passage:` beim Indexieren und
`query:` beim Suchen, die der Wrapper automatisch setzt.

In [ ]:
class E5OpenAIEmbeddings(OpenAIEmbeddings):
    """E5-Präfixe für OpenAI-kompatible Endpunkte (LM Studio)."""

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents(["passage: " + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query("query: " + text)


def build_embeddings():
    """Erzeugt das Embedding-Modell passend zu EMBEDDING_BACKEND."""
    if EMBEDDING_BACKEND == "lmstudio":
        return E5OpenAIEmbeddings(
            model=EMBEDDING_MODELS["lmstudio"],
            api_key="lm-studio",
            base_url=LM_STUDIO_URL,
            check_embedding_ctx_length=False,
        )

    if EMBEDDING_BACKEND == "huggingface":
        # Import erst hier, damit LM-Studio-Nutzer das Paket nicht brauchen
        from langchain_huggingface import HuggingFaceEmbeddings

        class E5HuggingFaceEmbeddings(HuggingFaceEmbeddings):
            """E5-Präfixe für lokal geladene sentence-transformers-Modelle."""

            def embed_documents(self, texts: list[str]) -> list[list[float]]:
                return super().embed_documents(["passage: " + t for t in texts])

            def embed_query(self, text: str) -> list[float]:
                return super().embed_query("query: " + text)

        return E5HuggingFaceEmbeddings(
            model_name=EMBEDDING_MODELS["huggingface"],
            cache_folder=MODEL_PATH,
        )

    raise ValueError(f"Unbekanntes EMBEDDING_BACKEND: {EMBEDDING_BACKEND}")

embeddings = build_embeddings()

vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    persist_directory=DB_DIR,
    embedding_function=embeddings,
)

print(f"✅ Vektordatenbank geladen – {vectorstore._collection.count()} Chunks verfügbar.")

## Reranking-Modell laden

Ein **Bi-Encoder** (unser E5-Modell) ist schnell, weil Query und Dokument getrennt
eingebettet werden. Ein **Cross-Encoder** ist langsamer, aber genauer: er bewertet Query
und Dokument *gemeinsam*.

Strategie: erst breit suchen, dann mit dem Cross-Encoder die besten Kandidaten auswählen.

> **Wichtig:** Unsere Dokumente sind auf Deutsch, wir brauchen also einen **multilingualen**
> Cross-Encoder. Ein rein englischer Reranker (z.B. `ms-marco-MiniLM`) liefert bei deutschen
> Texten systematisch zu niedrige Scores.

> **Ebenso wichtig:** MS-MARCO-Cross-Encoder geben ohne `activation_fn` **rohe Logits**
> zurück – Werte etwa zwischen −10 und +10. Die Schwelle `RELEVANCE_THRESHOLD = 0.3` wäre
> darauf angewandt sinnlos. Mit der Sigmoid liegen die Scores in [0, 1] und lassen sich als
> Relevanz-Wahrscheinlichkeit lesen.

In [ ]:
# activation_fn=Sigmoid ist hier entscheidend: ohne sie liefern MS-MARCO-
# Cross-Encoder rohe Logits (etwa -10 … +10). Mit Sigmoid sind die Scores
# Wahrscheinlichkeiten in [0, 1] – und RELEVANCE_THRESHOLD wird interpretierbar.
reranker = CrossEncoder(
    RERANKER_MODEL,
    max_length=512,
    activation_fn=torch.nn.Sigmoid(),
)

print(f"✅ Reranker geladen: {RERANKER_MODEL}")

## LLM konfigurieren

In [ ]:
if LLM_BACKEND not in LLM_CONFIG:
    raise ValueError(f"Unbekanntes LLM_BACKEND: {LLM_BACKEND}")

cfg = LLM_CONFIG[LLM_BACKEND]

if not cfg["api_key"]:
    raise ValueError(f"Kein API-Key für '{LLM_BACKEND}' – bitte in der Konfigurationszelle eintragen.")

llm = ChatOpenAI(
    model=cfg["model"],
    api_key=cfg["api_key"],
    base_url=cfg["base_url"],
    max_tokens=5000,
    temperature=0,          # Für RAG: keine Kreativität, sondern Fakten
)

print(f"✅ LLM bereit: {cfg['model']} ({LLM_BACKEND}).")

## State – zwei Fragen statt einer

Der wichtigste Unterschied zu Notebook 2:

| Feld | Bedeutung | Wer schreibt es? |
|---|---|---|
| `question` | Die Frage des Menschen – **bleibt unverändert** | niemand |
| `search_query` | Die Anfrage, mit der gesucht wird | `reformulate` |

Das ist keine Kosmetik. Wenn der Graph reformuliert, sucht er mit einer maschinell
erzeugten Query weiter – aber beantworten soll er immer noch das, was der Mensch gefragt
hat. Steckte beides in einem Feld, würde `generate` am Ende die Suchanfrage beantworten
statt die Frage.

Dazu kommen:
- `rerank_scores` – die Bewertungen des Cross-Encoders
- `retry_count` – zählt die Reformulierungen (verhindert Endlosschleifen)

In [ ]:
class GraphState(TypedDict):
    question:      str          # Frage des Menschen – bleibt unverändert
    search_query:  str          # Anfrage an die Suche – wird reformuliert
    context:       List[str]
    metadata:      List[dict]
    rerank_scores: List[float]
    answer:        str
    token_usage:   dict
    retry_count:   int

## Nodes – die Bausteine des Graphen

Jede Node ist eine gewöhnliche Python-Funktion: sie bekommt den State und gibt ein Dict
mit den Feldern zurück, die sie verändert hat.

In [ ]:
# --- NODE: RETRIEVE ---

def retrieve(state: GraphState) -> dict:
    """Holt Kandidaten-Chunks aus der Vektordatenbank (breite Suche)."""
    # Gesucht wird mit search_query; beim ersten Durchlauf ist das die Nutzerfrage
    query = state.get("search_query") or state["question"]

    print(f"--- RETRIEVE (Versuch {state.get('retry_count', 0) + 1}) ---")
    print(f"    Suchanfrage: {query}")

    docs = vectorstore.similarity_search(query, k=RETRIEVE_K)

    context  = []
    metadata = []

    for i, doc in enumerate(docs):
        context.append(doc.page_content)
        source_file = os.path.basename(doc.metadata.get("source", "Unbekannt"))
        page_num    = doc.metadata.get("page", 0) + 1
        metadata.append({"id": i + 1, "source": source_file, "page": page_num})

    print(f"    → {len(docs)} Kandidaten gefunden")
    return {"context": context, "metadata": metadata}

In [ ]:
# --- NODE: RERANK ---

def rerank(state: GraphState) -> dict:
    """Bewertet die Kandidaten mit einem Cross-Encoder und behält die Top-N."""
    print("--- RERANK ---")

    # Bewertet wird gegen die Suchanfrage, mit der die Kandidaten gefunden wurden
    query = state.get("search_query") or state["question"]

    # Cross-Encoder erwartet Paare aus (Query, Passage)
    pairs  = [(query, chunk) for chunk in state["context"]]
    scores = reranker.predict(pairs)

    # Nach Score sortieren (absteigend) und Top-N behalten
    scored_items = sorted(
        zip(scores, state["context"], state["metadata"]),
        key=lambda x: x[0],
        reverse=True,
    )
    top_items = scored_items[:RERANK_TOP_N]

    # IDs neu vergeben (1-basiert)
    reranked_context  = []
    reranked_metadata = []
    reranked_scores   = []

    for new_id, (score, text, meta) in enumerate(top_items, start=1):
        reranked_context.append(text)
        reranked_metadata.append({**meta, "id": new_id})
        reranked_scores.append(float(score))

    print(f"    → Top-{RERANK_TOP_N} Scores: {[f'{s:.3f}' for s in reranked_scores]}")

    return {
        "context":       reranked_context,
        "metadata":      reranked_metadata,
        "rerank_scores": reranked_scores,
    }

In [ ]:
# --- NODE: GENERATE ---

ANSWER_PROMPT = ChatPromptTemplate.from_template("""\
Du bist ein präziser Assistent. Beantworte die Frage NUR basierend auf dem KONTEXT.

REGELN:
1. Verweise im Text deiner Antwort auf die Abschnitte, z.B. [1] oder [Quelle: Datei.pdf, S. 5].
2. Wenn die Info nicht im Kontext ist, sag es offen.
3. Erfinde KEINE Fakten.

KONTEXT:
{context}

FRAGE: {question}
""")


def generate(state: GraphState) -> dict:
    """Erzeugt eine Antwort auf Basis der besten Kontext-Chunks."""
    print("--- GENERATE ---")

    formatted_context = ""
    for i, text in enumerate(state["context"]):
        meta  = state["metadata"][i]
        score = state["rerank_scores"][i]
        formatted_context += (
            f"\n--- ABSCHNITT {meta['id']} "
            f"(Quelle: {meta['source']}, Seite {meta['page']}, "
            f"Relevanz: {score:.3f}) ---\n"
            f"{text}\n"
        )

    # Beantwortet wird IMMER die Frage des Menschen – nie die Suchanfrage
    chain    = ANSWER_PROMPT | llm
    response = chain.invoke({"context": formatted_context, "question": state["question"]})
    usage    = response.response_metadata.get("token_usage", {})

    return {"answer": response.content, "token_usage": usage}

In [ ]:
# --- NODE: REFORMULATE ---

REFORMULATE_PROMPT = ChatPromptTemplate.from_template("""\
Die folgende Suchanfrage hat keine ausreichend relevanten Ergebnisse geliefert.
Formuliere eine NEUE, andere Suchanfrage – nutze Synonyme, Fachbegriffe oder einen
präziseren Kern. Antworte NUR mit der neuen Suchanfrage, ohne Erklärung.

Frage des Nutzers:      {question}
Bisherige Suchanfrage:  {search_query}
""")


def reformulate(state: GraphState) -> dict:
    """Erzeugt eine neue Suchanfrage – die Nutzerfrage bleibt unangetastet."""
    print("--- REFORMULATE ---")

    chain    = REFORMULATE_PROMPT | llm
    response = chain.invoke({
        "question":     state["question"],
        "search_query": state.get("search_query") or state["question"],
    })
    new_query = response.content.strip()

    retry_count = state.get("retry_count", 0) + 1
    print(f"    → Neue Suchanfrage: '{new_query}' (Versuch {retry_count})")

    # Nur search_query wird ersetzt – question bleibt, wie der Mensch sie gestellt hat
    return {"search_query": new_query, "retry_count": retry_count}

## Bedingte Kante – das Herzstück

`check_relevance` ist keine Node, sondern eine **Routing-Funktion**: sie verändert den
State nicht, sondern gibt nur einen String zurück, der bestimmt, welche Kante als nächstes
genommen wird. Genau das kann eine lineare Chain nicht.

In [ ]:
# --- CONDITIONAL EDGE ---

def check_relevance(state: GraphState) -> str:
    """Entscheidet, ob der Kontext relevant genug ist oder reformuliert werden muss."""
    best_score  = max(state["rerank_scores"]) if state["rerank_scores"] else 0.0
    retry_count = state.get("retry_count", 0)

    print("--- CHECK RELEVANCE ---")
    print(f"    Bester Score: {best_score:.3f} (Schwelle: {RELEVANCE_THRESHOLD})")
    print(f"    Bisherige Versuche: {retry_count} / {MAX_RETRIES}")

    if best_score >= RELEVANCE_THRESHOLD:
        print("    → RELEVANT – weiter zu Generate")
        return "relevant"

    if retry_count < MAX_RETRIES:
        print("    → NICHT RELEVANT – Suchanfrage wird reformuliert")
        return "not_relevant"

    print("    → NICHT RELEVANT, aber max. Versuche erreicht – Antwort mit letztem Ergebnis")
    return "relevant"   # Fallback: lieber eine schwache Antwort als gar keine

## Graph zusammenbauen

In [ ]:
# --- GRAPH ZUSAMMENBAUEN ---

workflow = StateGraph(GraphState)

# Knoten registrieren
workflow.add_node("retrieve_node",    retrieve)
workflow.add_node("rerank_node",      rerank)
workflow.add_node("generate_node",    generate)
workflow.add_node("reformulate_node", reformulate)

# Kanten definieren
workflow.add_edge(START, "retrieve_node")
workflow.add_edge("retrieve_node", "rerank_node")

# ⭐ Die bedingte Kante: check_relevance entscheidet den Weg
workflow.add_conditional_edges(
    "rerank_node",           # Nach diesem Knoten ...
    check_relevance,         # ... wird diese Funktion aufgerufen ...
    {                        # ... und ihr Rückgabewert bestimmt den nächsten Knoten:
        "relevant":     "generate_node",
        "not_relevant": "reformulate_node",
    },
)

# Der Zyklus: reformulate → retrieve (und von dort wieder rerank → check)
workflow.add_edge("reformulate_node", "retrieve_node")
workflow.add_edge("generate_node", END)

# Graph kompilieren
app = workflow.compile()

## Graph visualisieren

Vergleiche das Diagramm mit dem aus Notebook 2 – der Unterschied ist der ganze Punkt
dieses Notebooks.

In [ ]:
def display_graph(graph_app):
    """Zeigt den LangGraph als Mermaid-Diagramm an und speichert die Syntax."""
    mermaid_code = graph_app.get_graph().draw_mermaid()

    # Diagramm im Notebook rendern (via mermaid.ink)
    encoded = base64.b64encode(mermaid_code.encode()).decode()
    display.display(display.Image(url=f"https://mermaid.ink/img/{encoded}"))

    # Mermaid-Syntax als Datei speichern (optional)
    with open("rag_graph.mmd", "w", encoding="utf-8") as f:
        f.write(mermaid_code)


display_graph(app)

## Gradio-Interface

In [ ]:
def chat_interface(question: str) -> str:
    """Verarbeitet eine Frage über den erweiterten RAG-Graphen."""
    result = app.invoke({
        "question":     question,
        "search_query": question,   # erster Versuch: wörtlich die Nutzerfrage
        "retry_count":  0,
    })

    answer = result["answer"]

    # Reranking-Info
    scores = result.get("rerank_scores", [])
    rerank_info = "\n\n---\n🎯 **Reranking-Scores (Top-Chunks):**\n"
    for meta, score in zip(result["metadata"], scores):
        rerank_info += f"- [{meta['id']}] {meta['source']} (S. {meta['page']}): {score:.3f}\n"

    # Token-Statistik
    usage = result.get("token_usage", {})
    token_info = (
        f"\n📊 **Token-Statistik:**\n"
        f"- Input: {usage.get('prompt_tokens', 'N/A')}\n"
        f"- Output: {usage.get('completion_tokens', 'N/A')}\n"
        f"- Gesamt: {usage.get('total_tokens', 'N/A')}"
    )

    # Retry-Info
    retries = result.get("retry_count", 0)
    retry_info = ""
    if retries > 0:
        retry_info = (
            f"\n\n🔄 Suchanfrage wurde {retries}× reformuliert.\n"
            f"- Zuletzt gesucht mit: {result.get('search_query', '–')}"
        )

    return answer + rerank_info + token_info + retry_info


demo = gr.Interface(
    fn=chat_interface,
    inputs="text",
    outputs="text",
    title="RAG mit Reranking & Relevanz-Gate",
    description="Fragen an deine Dokumente – mit Cross-Encoder-Reranking und automatischer Reformulierung.",
    flagging_mode="never",
)

demo.launch()

## Zum Nachdenken

- Wie verändert das Reranking die Antwortqualität gegenüber Notebook 2?
- Was passiert bei `RELEVANCE_THRESHOLD = 0.8`? Bei `0.01`?
- Setze `activation_fn` versuchsweise auf `torch.nn.Identity()` und schau dir die Scores an.
  Was macht das mit dem Gate?
- Die Reformulierung sieht nur die **letzte** Suchanfrage. Was kann dabei schiefgehen,
  wenn `MAX_RETRIES` größer als 2 ist? → Notebook 3b